# Mol* visualization catalog

Progress dashboard for the molstarLib migration (legacy `deeporigin-molstar` → hosted
`molstarLib` at `os.dev.deeporigin.io`).

| # | Visualization | Status |
|---|---------------|--------|
| 1 | Protein structure | ✅ migrated |
| 2 | Protein + binding pockets | ✅ migrated |
| 3 | Single ligand 3D | ⏳ legacy |
| 4 | Ligand set 3D | ⏳ legacy |
| 5 | Protein + docked poses | ⏳ legacy |
| 6 | Docking search box | ⏳ legacy |
| 7 | MD trajectory | ⏳ legacy |

**Requires:** `uv sync --extra tools` for sections 3–7 (legacy molstar). Sections 1–2
use the new hosted bundle only.

In [ ]:
from dotenv import load_dotenv

load_dotenv()

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path

from deeporigin.drug_discovery import BRD_DATA_DIR, Ligand, LigandSet, Pocket, Protein
from deeporigin.viz.molstar_html import MOLSTAR_JS_URL


def _find_repo_root() -> Path:
    """Return the CLI repo root whether the notebook cwd is repo root or docs/notebooks/."""
    path = Path.cwd().resolve()
    for candidate in (path, *path.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "tests").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not find CLI repo root. Run this notebook from the cli repo "
        "(repo root or docs/notebooks/dirty/)."
    )


REPO_ROOT = _find_repo_root()
POCKET_FIXTURE = REPO_ROOT / "tests/fixtures/files/pocketfinder/pocket_1.pdb"

print(f"Repo root: {REPO_ROOT}")
print(f"Mol* bundle URL: {MOLSTAR_JS_URL}")
print(f"Pocket fixture: {POCKET_FIXTURE} ({'ok' if POCKET_FIXTURE.is_file() else 'missing'})")

## 1. Protein structure — ✅ migrated

Entry point: `Protein.show()` (no pockets or poses).

**Check:** cartoon representation, Mol* UI loads, no iframe console errors.

In [ ]:
protein = Protein.from_file(BRD_DATA_DIR / "brd.pdb")
protein.show()

## 2. Protein + binding pockets — ✅ migrated

Entry point: `Protein.show(pockets=...)`.

**Check:** semi-transparent pocket surfaces (alpha ~0.7), protein cartoon with faint
surface (alpha ~0.1), pocket colors match `Pocket.color`.

In [ ]:
pocket = Pocket.from_pdb_file(POCKET_FIXTURE, name="pocket-1", color="red")
pocket.box_size_x = pocket.box_size_y = pocket.box_size_z = 30.0
pocket.get_center()

protein = Protein.from_file(BRD_DATA_DIR / "brd.pdb")
protein.show(pockets=[pocket])

## 3. Single ligand 3D — ⏳ legacy

Entry point: `Ligand.show()`.

**Check:** ball-and-stick ligand in Mol* viewer. Still uses `deeporigin-molstar` until phase 3.

In [ ]:
ligand = Ligand.from_sdf(BRD_DATA_DIR / "brd-2.sdf")
ligand.show()

## 4. Ligand set 3D — ⏳ legacy

Entry point: `LigandSet.show()`.

**Check:** multiple ligands rendered together. Still uses `deeporigin-molstar` until phase 3.

In [ ]:
ligands = LigandSet.from_dir(BRD_DATA_DIR)
ligands.show()

## 5. Protein + docked poses — ⏳ legacy

Entry point: `Protein.show(poses=...)`.

**Check:** protein cartoon with docked ligand poses overlaid. Uses BRD ligands as stand-in poses.

In [ ]:
protein = Protein.from_file(BRD_DATA_DIR / "brd.pdb")
poses = LigandSet.from_dir(BRD_DATA_DIR)
protein.show(poses=poses)

## 6. Docking search box — ⏳ legacy

Entry point: `Docking.show_box()`.

**Check:** protein with wireframe bounding box from pocket center and box dimensions.

In [ ]:
from deeporigin.drug_discovery import Docking

protein = Protein.from_file(BRD_DATA_DIR / "brd.pdb")
# Docking() requires protein.id; use a placeholder for local visualization only.
if protein.id is None:
    protein.id = "notebook-demo-protein"

ligand = Ligand.from_sdf(BRD_DATA_DIR / "brd-2.sdf")
pocket = Pocket.from_pdb_file(POCKET_FIXTURE, name="pocket-1")
pocket.box_size_x = pocket.box_size_y = pocket.box_size_z = 30.0
pocket.get_center()

docking = Docking(protein=protein, pocket=pocket, ligand=ligand)
docking.show_box()

## 7. MD trajectory — ⏳ legacy

Entry point: `ABFE.show_trajectory()`.

**Check:** trajectory playback in Mol* with protein + XTC frames.

Requires a **completed ABFE execution** with trajectory paths in results. Run the
[ABFE notebook](./abfe.ipynb) first, then uncomment and set `abfe` below.

In [ ]:
# from deeporigin.drug_discovery import ABFE
#
# abfe = ABFE.from_id("<completed-abfe-execution-id>")
# abfe.show_trajectory(step="binding", window=1)